# Compare NeuralLaplace encoder train modes

Этот ноутбук сравнивает 3 варианта `ENCODER_TRAIN_DATA_MODE`:
- `all`
- `random_half`
- `time_half`


## 1) Imports


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import polars as pl

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from run_benchmark import load_dataset
from bandit_benchmark import (
    NeuralLaplaceThompsonViaBayesianLogRegPolicy,
    RandomPolicy,
    core_scenarios,
    default_five_scenarios,
    run_scenarios,
)
from prepare_datasets import stage1_make_splits, stage2_scale_features


## 2) Config


In [ ]:
DATA_PATH = ROOT / 'data' / 'events.tsv'
DATASET_NAME = DATA_PATH.stem
ARTIFACTS_DIR = ROOT / 'artifacts' / DATASET_NAME
DATASETS_DIR = ARTIFACTS_DIR / 'datasets'

PREPARED_TRAIN_PATH = DATASETS_DIR / 'train_variant_1_scaled.parquet'
PREPARED_TEST_PATH = DATASETS_DIR / 'test_variant_1_scaled.parquet'

USE_PREPARED_SPLITS = True
TRAIN_DAYS = 1
SEED = 42
FULL_SCENARIOS = False

# neural params
NEURAL_HIDDEN_DIMS = [64, 32]
NN_EPOCHS = 10
NN_BATCH_SIZE = 256


## 3) Load / prepare dataset


In [ ]:
if USE_PREPARED_SPLITS and PREPARED_TRAIN_PATH.exists() and PREPARED_TEST_PATH.exists():
    train_df = pl.read_parquet(PREPARED_TRAIN_PATH)
    test_df = pl.read_parquet(PREPARED_TEST_PATH)
    print('loaded prepared splits')
else:
    train_s1, test_s1 = stage1_make_splits(str(DATA_PATH), str(DATASETS_DIR), TRAIN_DAYS, SEED)
    train_final, test_final = stage2_scale_features(train_s1, test_s1, str(DATASETS_DIR))
    train_df = pl.read_parquet(train_final)
    test_df = pl.read_parquet(test_final)

print('train:', train_df.height, 'test(random only):', test_df.height)


## 4) Compare 3 encoder modes


In [ ]:
policy_factories = {
    'neural_all': lambda: NeuralLaplaceThompsonViaBayesianLogRegPolicy(
        seed=SEED,
        network_architecture=NEURAL_HIDDEN_DIMS,
        encoder_train_data_mode='all',
        nn_epochs=NN_EPOCHS,
        nn_batch_size=NN_BATCH_SIZE,
    ),
    'neural_random_half': lambda: NeuralLaplaceThompsonViaBayesianLogRegPolicy(
        seed=SEED,
        network_architecture=NEURAL_HIDDEN_DIMS,
        encoder_train_data_mode='random_half',
        nn_epochs=NN_EPOCHS,
        nn_batch_size=NN_BATCH_SIZE,
    ),
    'neural_time_half': lambda: NeuralLaplaceThompsonViaBayesianLogRegPolicy(
        seed=SEED,
        network_architecture=NEURAL_HIDDEN_DIMS,
        encoder_train_data_mode='time_half',
        nn_epochs=NN_EPOCHS,
        nn_batch_size=NN_BATCH_SIZE,
    ),
    # optional reference baseline
    'random': lambda: RandomPolicy(seed=SEED),
}

scenarios = default_five_scenarios() if FULL_SCENARIOS else core_scenarios()

result = run_scenarios(
    train_df=train_df,
    test_df=test_df,
    policy_factories=policy_factories,
    scenarios=scenarios,
    env_reward=None,
    show_progress=True,
)

metrics_df = result['metrics']
history_df = result['history']

display(metrics_df.sort_values(['scenario', 'ips_ctr'], ascending=[True, False]).reset_index(drop=True))


## 5) Focus table: only neural variants


In [ ]:
neural_metrics = metrics_df[metrics_df['algo'].str.startswith('neural_')].copy()
display(neural_metrics.sort_values(['scenario', 'ips_ctr'], ascending=[True, False]).reset_index(drop=True))


## 6) Plot IPS CTR by mode


In [ ]:
if not neural_metrics.empty:
    for scenario_name, part in neural_metrics.groupby('scenario'):
        fig, ax = plt.subplots(figsize=(8, 4))
        part = part.sort_values('ips_ctr', ascending=False)
        ax.bar(part['algo'], part['ips_ctr'])
        ax.set_title(f'{scenario_name}: IPS CTR by encoder mode')
        ax.set_xlabel('model')
        ax.set_ylabel('ips_ctr')
        ax.tick_params(axis='x', rotation=25)
        ax.grid(True, alpha=0.3)
        fig.tight_layout()
        plt.show()
else:
    print('No neural metrics to plot.')


## 7) Save comparison artifacts


In [ ]:
OUT_DIR = ARTIFACTS_DIR / 'encoder_mode_compare'
OUT_DIR.mkdir(parents=True, exist_ok=True)

metrics_path = OUT_DIR / 'metrics_compare.csv'
history_path = OUT_DIR / 'history_compare.csv'

metrics_df.to_csv(metrics_path, index=False)
history_df.to_csv(history_path, index=False)

print('saved:', metrics_path)
print('saved:', history_path)
